# Data Analysis

Concrete version
Step 1 — salary score
jobs_df["avg_salary"] = (
    jobs_df["salary_minimum"] + jobs_df["salary_maximum"]
) / 2
Step 2 — normalize
jobs_df["salary_score"] = (
    jobs_df["avg_salary"] - jobs_df["avg_salary"].min()
) / (jobs_df["avg_salary"].max() - jobs_df["avg_salary"].min())
Step 3 — optional stability
jobs_df["stability_score"] = (jobs_df["contract_type"] == "Permanent").astype(int)

⚠️ But since ~44% Unknown:
👉 keep this weight LOW or ignore

Step 4 — final job score
jobs_df["job_score"] = jobs_df["salary_score"]

## Load Data

In [96]:
import pandas as pd

jobs_df = pd.read_pickle("../../data/jobs_cleaned.pkl")
combined_df = pd.read_pickle("../../data/combined_courses_cleaned.pkl")

In [97]:
jobs_df.head(1)

,uuid,salary_minimum,salary_maximum,posting_date,expiry_date,has_flexible_work,ssoc_3d,contract_type,work_type,avg_salary,skills_clean,num_skills
0,2cc5117f205f59b400cb529d2253651a,3000,6000,2026-01-29,2026-02-28,False,243,Permanent,Full Time,4500.0,"[shipyards, technical sales, project bidding, ...",19


In [98]:
combined_df.head(1)

,s/n,code,title,department,description,university,description_clean,description_chunks,skills_embedding,hard_skills,soft_skills,num_skills,num_hard_skills,num_soft_skills
0,1,ACC1701,Accounting for Decision Makers,Accounting,The course provides an introduction to account...,NUS,the course provides an introduction to account...,[the course provides an introduction to accoun...,"[accounting, financial reporting, financial ac...","[accounting, financial reporting, financial ac...",[],5,5,0


# Job Score

## Salary

In [99]:
# Salary
jobs_df["salary_range"] = (
    jobs_df["salary_maximum"] - jobs_df["salary_minimum"]
)

jobs_df["salary_range_pct"] = jobs_df["salary_range"] / jobs_df["avg_salary"]

In [100]:
jobs_df["salary_score"] = (
    jobs_df["avg_salary"] - jobs_df["avg_salary"].min()
) / (jobs_df["avg_salary"].max() - jobs_df["avg_salary"].min())

In [101]:
jobs_df.head(1)

,uuid,salary_minimum,salary_maximum,posting_date,expiry_date,has_flexible_work,ssoc_3d,contract_type,work_type,avg_salary,skills_clean,num_skills,salary_range,salary_range_pct,salary_score
0,2cc5117f205f59b400cb529d2253651a,3000,6000,2026-01-29,2026-02-28,False,243,Permanent,Full Time,4500.0,"[shipyards, technical sales, project bidding, ...",19,3000,0.666667,0.187466


In [102]:
jobs_df["work_type"].value_counts()

work_type
Full Time    8154
Part Time     782
Name: count, dtype: int64

## Data Analysis

### Alignment

In [103]:
def alignment_score(course_skills, job_skills):
    if not course_skills or not job_skills:
        return 0.0
    
    course_set = set(course_skills)
    job_set = set(job_skills)
    
    return len(course_set & job_set) / len(job_set)

In [104]:
jobs_df_base = jobs_df.copy()
jobs_df_work = jobs_df_base.copy()

In [105]:
course = combined_df.iloc[0]
course_skills = course["skills_embedding"]

jobs_df_work = jobs_df.copy()

jobs_df_work["alignment_score"] = jobs_df_work["skills_clean"].apply(
    lambda job_skills: alignment_score(course_skills, job_skills)
)

jobs_df_work["final_score"] = (
    jobs_df_work["alignment_score"] * (0.7 + 0.3 * jobs_df_work["salary_score"])
)

In [106]:
jobs_df_work.sort_values("alignment_score", ascending=False)[
    ["uuid", "avg_salary", "skills_clean", "alignment_score"]
].head(5)

,uuid,avg_salary,skills_clean,alignment_score
3441,75ee5de0e5664e19124899216f55d9d2,3200.0,"[administration, purchasing, accounts payable,...",0.250000
10425,d1c33c13e91397a16b7ea61f43ca56f1,3200.0,"[accruals, accounts receivable, administration...",0.222222
4512,e6c008399e40a6d3ec3f7ef4571e16cf,3000.0,"[ad hoc reporting, general ledger, management,...",0.222222
22345,c82a1f801139f0cd805d53b2417f0e59,3900.0,"[risk assessment, microsoft office, data analy...",0.222222
15358,730ce454cc30370b3efdacb4b59be78b,3200.0,"[financial statements, inventory, administrati...",0.222222


In [107]:
jobs_df_work.sort_values("final_score", ascending=False)[
    ["uuid", "avg_salary", "skills_clean", "alignment_score", "final_score"]
].head(5)

,uuid,avg_salary,skills_clean,alignment_score,final_score
3441,75ee5de0e5664e19124899216f55d9d2,3200.0,"[administration, purchasing, accounts payable,...",0.250000,0.184997
22345,c82a1f801139f0cd805d53b2417f0e59,3900.0,"[risk assessment, microsoft office, data analy...",0.222222,0.166387
10425,d1c33c13e91397a16b7ea61f43ca56f1,3200.0,"[accruals, accounts receivable, administration...",0.222222,0.164442
15358,730ce454cc30370b3efdacb4b59be78b,3200.0,"[financial statements, inventory, administrati...",0.222222,0.164442
4512,e6c008399e40a6d3ec3f7ef4571e16cf,3000.0,"[ad hoc reporting, general ledger, management,...",0.222222,0.163886


In [108]:
def overlapping_skills(course_skills, job_skills):
    return sorted(set(course_skills) & set(job_skills))

jobs_df_work["overlap_skills"] = jobs_df_work["skills_clean"].apply(
    lambda job_skills: overlapping_skills(course_skills, job_skills)
)

In [109]:
jobs_df_work.sort_values("final_score", ascending=False)[
    ["uuid", "avg_salary", "skills_clean", "overlap_skills", "alignment_score", "final_score"]
].head(5)

,uuid,avg_salary,skills_clean,overlap_skills,alignment_score,final_score
3441,75ee5de0e5664e19124899216f55d9d2,3200.0,"[administration, purchasing, accounts payable,...","[accounting, financial reporting]",0.250000,0.184997
22345,c82a1f801139f0cd805d53b2417f0e59,3900.0,"[risk assessment, microsoft office, data analy...","[accounting, financial reporting]",0.222222,0.166387
10425,d1c33c13e91397a16b7ea61f43ca56f1,3200.0,"[accruals, accounts receivable, administration...","[accounting, financial reporting]",0.222222,0.164442
15358,730ce454cc30370b3efdacb4b59be78b,3200.0,"[financial statements, inventory, administrati...","[accounting, financial reporting]",0.222222,0.164442
4512,e6c008399e40a6d3ec3f7ef4571e16cf,3000.0,"[ad hoc reporting, general ledger, management,...","[accounting, financial reporting]",0.222222,0.163886


In [110]:
for i in [0, 500, 3000]:
    jobs_df_work = jobs_df.copy()

    course = combined_df.iloc[i]
    course_skills = course["skills_embedding"]

    jobs_df_work["alignment_score"] = jobs_df_work["skills_clean"].apply(
        lambda job_skills: alignment_score(course_skills, job_skills)
    )

    jobs_df_work["overlap_skills"] = jobs_df_work["skills_clean"].apply(
        lambda job_skills: sorted(set(course_skills) & set(job_skills))
    )

    print(course["code"], course["title"])
    print(
        jobs_df_work.sort_values("alignment_score", ascending=False)[
            ["uuid", "avg_salary", "overlap_skills", "alignment_score"]
        ].head(5)
    )
    print("-" * 80)

ACC1701 Accounting for Decision Makers
                                   uuid  avg_salary  \
3441   75ee5de0e5664e19124899216f55d9d2      3200.0   
10425  d1c33c13e91397a16b7ea61f43ca56f1      3200.0   
4512   e6c008399e40a6d3ec3f7ef4571e16cf      3000.0   
22345  c82a1f801139f0cd805d53b2417f0e59      3900.0   
15358  730ce454cc30370b3efdacb4b59be78b      3200.0   

                          overlap_skills  alignment_score  
3441   [accounting, financial reporting]         0.250000  
10425  [accounting, financial reporting]         0.222222  
4512   [accounting, financial reporting]         0.222222  
22345  [accounting, financial reporting]         0.222222  
15358  [accounting, financial reporting]         0.222222  
--------------------------------------------------------------------------------
CE4221 Design of Land Transport Infrastructures
                                   uuid  avg_salary          overlap_skills  \
7308   3b5e13bbb7251b7835b70331407fe727      4875.0        [ur

### Compute course-level scores

In [111]:
combined_df_base = combined_df.copy()

In [112]:
combined_df_work = combined_df_base.copy()

def compute_course_scores(course_skills):
    temp = jobs_df.copy()

    temp["alignment_score"] = temp["skills_clean"].apply(
        lambda job_skills: alignment_score(course_skills, job_skills)
    )

    temp["final_score"] = temp["alignment_score"] * (0.7 + 0.3 * temp["salary_score"])

    return {
        "avg_alignment": temp["alignment_score"].mean(),
        "max_alignment": temp["alignment_score"].max(),
        "avg_final_score": temp["final_score"].mean(),
        "max_final_score": temp["final_score"].max()
    }

results = combined_df_work["skills_embedding"].apply(compute_course_scores)
results_df = pd.DataFrame(list(results))

combined_df_work = pd.concat([combined_df_work, results_df], axis=1)

In [114]:
combined_df_work.sort_values("avg_alignment", ascending=False)[
    ["code", "title", "avg_alignment"]
].head(10)

,code,title,avg_alignment
9030,CS2101,CS2101 PUBLIC RELATIONS WRITING,0.056520
1821,ES2002,Business Communication for Leaders (BBA),0.056459
5310,NM3242,Organisational Communication and Leadership,0.056371
8309,YSS3285,Organisational Psychology,0.056371
5294,NM2219,Principles of Communication Management,0.052665
9023,CS2055,CS2055 ORGANISATIONAL COMMUNICATION,0.052258
7472,UTS2416,Communicating with Communities in the 21st cen...,0.052258
7303,UTC2413,Communicating with Communities in the 21st cen...,0.052258
524,CFA3105,Leading and Directing in Student Music Groups,0.052252
9071,CS4196,CS4196 IMPRESSION MANAGEMENT & SELF-PRESENTATI...,0.052170


In [115]:
combined_df_work.sort_values("avg_final_score", ascending=False)[
    ["code", "title", "avg_final_score"]
].head(10)

,code,title,avg_final_score
9030,CS2101,CS2101 PUBLIC RELATIONS WRITING,0.042169
1821,ES2002,Business Communication for Leaders (BBA),0.042124
5310,NM3242,Organisational Communication and Leadership,0.042058
8309,YSS3285,Organisational Psychology,0.042058
5294,NM2219,Principles of Communication Management,0.039305
7472,UTS2416,Communicating with Communities in the 21st cen...,0.038999
9023,CS2055,CS2055 ORGANISATIONAL COMMUNICATION,0.038999
7303,UTC2413,Communicating with Communities in the 21st cen...,0.038999
524,CFA3105,Leading and Directing in Student Music Groups,0.038994
9071,CS4196,CS4196 IMPRESSION MANAGEMENT & SELF-PRESENTATI...,0.038933


In [116]:
combined_df_work.groupby("department")["avg_alignment"].mean().sort_values(ascending=False)

department
Communications and New Media                         0.014957
Centre for Future-ready Grads                        0.014938
School of Computer Science and Engineering (SCSE)    0.013502
Management and Organisation                          0.011139
Centre for Language Studies                          0.010716
                                                       ...   
Biomedical Sciences (Biological Sciences)            0.000046
Pathology                                            0.000000
Career & Attachment Office                           0.000000
FoD Dean's Office                                    0.000000
Residential Colleges                                 0.000000
Name: avg_alignment, Length: 129, dtype: float64

In [117]:
exclude = [
    "Career & Attachment Office",
    "FoD Dean's Office",
    "Residential Colleges"
]

combined_df_work = combined_df_work[
    ~combined_df_work["department"].isin(exclude)
]

In [118]:
combined_df_work.sort_values("avg_alignment", ascending=False)[
    ["code", "title", "department", "avg_alignment"]
].head(10)

,code,title,department,avg_alignment
9030,CS2101,CS2101 PUBLIC RELATIONS WRITING,School of Computer Science and Engineering (SCSE),0.056520
1821,ES2002,Business Communication for Leaders (BBA),Center for Engl Lang Comms,0.056459
8309,YSS3285,Organisational Psychology,Yale-NUS College,0.056371
5310,NM3242,Organisational Communication and Leadership,Communications and New Media,0.056371
5294,NM2219,Principles of Communication Management,Communications and New Media,0.052665
9023,CS2055,CS2055 ORGANISATIONAL COMMUNICATION,School of Computer Science and Engineering (SCSE),0.052258
7303,UTC2413,Communicating with Communities in the 21st cen...,College of Alice and Peter Tan,0.052258
7472,UTS2416,Communicating with Communities in the 21st cen...,College of Alice and Peter Tan,0.052258
524,CFA3105,Leading and Directing in Student Music Groups,YSTCM Dean's Office,0.052252
9071,CS4196,CS4196 IMPRESSION MANAGEMENT & SELF-PRESENTATI...,School of Computer Science and Engineering (SCSE),0.052170


In [119]:
combined_df_work.sort_values("avg_alignment")[[
    "code", "title", "department", "avg_alignment"
]].head(10)

,code,title,department,avg_alignment
5255,NHS3006,Topics in Humanities and Social Sciences,NUS College Dean's Office,0.0
8891,CB1103,CB1103 ORGANIC CHEMISTRY FOR ENGINEERS,"School of Chemistry, Chemical Engineering & Bi...",0.0
8893,CB1131,CB1131 INTRODUCTION TO BIOMOLECULAR ENGINEERING,"School of Chemistry, Chemical Engineering & Bi...",0.0
8894,CB4001,CB4001 MICROFLUIDICS & ITS APPLICATIONS,"School of Chemistry, Chemical Engineering & Bi...",0.0
4655,MLE4228,Robotic Materials,Materials Science and Engineering,0.0
4652,MLE4224,Degradation and Failure of Materials,Materials Science and Engineering,0.0
4649,MLE4220,Two-Dimensional Materials,Materials Science and Engineering,0.0
4643,MLE4211,Nanoelectronics and information technology,Materials Science and Engineering,0.0
8890,CB1102,"CB1102 INTRO TO CHEMISTRY, CHEM ENG & BIOMED ENG","School of Chemistry, Chemical Engineering & Bi...",0.0
4642,MLE4210,Materials for energy storage and conversion,Materials Science and Engineering,0.0


In [120]:
combined_df_work.groupby("department")["avg_alignment"]\
.mean().sort_values(ascending=False).head(10)

department
Communications and New Media                         0.014957
Centre for Future-ready Grads                        0.014938
School of Computer Science and Engineering (SCSE)    0.013502
Management and Organisation                          0.011139
Centre for Language Studies                          0.010716
Center for Engl Lang Comms                           0.010333
EngrgDesignandInnovationCentre                       0.009108
Linguistics & Multilingual Studies                   0.007685
NUS Medicine Dean's Office                           0.007361
College of Alice and Peter Tan                       0.007070
Name: avg_alignment, dtype: float64

In [121]:
combined_df_work.groupby("department")["avg_alignment"]\
.mean().sort_values().head(10)

department
Pathology                                    0.000000
Biomedical Sciences (Biological Sciences)    0.000046
NUS Libraries                                0.000082
Office of Student Affairs                    0.000090
English (Humanities)                         0.000114
Pharmacology                                 0.000249
Biological Sciences (general)                0.000287
Anatomy                                      0.000293
Microbiology and Immunology                  0.000306
History                                      0.000307
Name: avg_alignment, dtype: float64